# NB03 — LCOE Model & Least-Cost Technology Selection
## Benin Least-Cost Electrification Analysis

**Base Scenario: SBEE National Expansion Programme**

| Parameter | Value | Source |
|---|---|---|
| Connection cost | $125/HH | ABERME Plan Directeur 2022 |
| MV backbone | Public infrastructure | OnSSET grid_price methodology |
| Grid generation cost | $0.08/kWh | SBEE bulk tariff |
| Solar MG min demand | 10,000 kWh/yr | ESMAP lower bound |
| Productive use uplift | ×1.30 | Health/school/cropland settlements |
| Demand source | VIDA satellite building footprints | Chi et al. 2022 |
| Demand growth | 3%/yr intensity + 2.7%/yr population | UN WPP 2024 |
| Discount rate | 10%/yr | ABERME Plan Directeur 2022 |
| Planning horizon | 15 years (2025–2040) | Benin SDG7 target |


In [ ]:
import sys, os
from pathlib import Path

project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import ast

from src.config import GRID, GENERAL, DEMAND, DISCOUNT_RATE, HORIZON
from src.costs.lcoe_calculator import npv as npv_fn
from src.costs.grid_extension import compute_grid_penalty
from src.costs.shs import add_shs_lcoe
from src.costs.mini_grid import add_minigrid_lcoe, MG_SUBTYPES

print('Imports OK ✓')
print(f'  Discount rate   : {DISCOUNT_RATE*100:.0f}%/yr')
print(f'  Planning horizon: {HORIZON} years')


## 1. Load Data
Loads the most recent `settlements_demand_*.geojson` from `data/processed/`.

In [ ]:
PROCESSED_DIR = Path('..') / 'data' / 'processed'
candidates = sorted(PROCESSED_DIR.glob('settlements_demand_*.geojson'), reverse=True)
if not candidates:
    raise FileNotFoundError('No settlements_demand_*.geojson — run NB02 first')

DEMAND_PATH = candidates[0]
print(f'Loading: {DEMAND_PATH.name}')
gdf = gpd.read_file(DEMAND_PATH)
print(f'Loaded {len(gdf):,} settlements | {len(gdf.columns)} columns')
print(f'  Electrified   : {(gdf["elec_status"]=="electrified").sum():,}')
print(f'  Unelectrified : {(gdf["elec_status"]=="unelectrified").sum():,}')

unelec_mask = gdf['elec_status'] == 'unelectrified'
elec_mask   = gdf['elec_status'] == 'electrified'


In [ ]:
# Ensure lat/lon columns exist for Streamlit map
if 'lat' not in gdf.columns or 'lon' not in gdf.columns:
    gdf['lon'] = gdf.geometry.centroid.x
    gdf['lat'] = gdf.geometry.centroid.y
    print(f"✓ lat/lon extracted from geometry")
    print(f"  lat range: {gdf['lat'].min():.2f} → {gdf['lat'].max():.2f}")
    print(f"  lon range: {gdf['lon'].min():.2f} → {gdf['lon'].max():.2f}")
else:
    print(f"✓ lat/lon already present")
    print(f"  lat range: {gdf['lat'].min():.2f} → {gdf['lat'].max():.2f}")
    print(f"  lon range: {gdf['lon'].min():.2f} → {gdf['lon'].max():.2f}")

In [ ]:
from src.config import GENERAL, DEMAND

T = GENERAL['planning_horizon_years']
growth = 1 + DEMAND.get('demand_growth_rate', 0.04)

print("Rebuilding demand_timeseries from demand_year0_kwh...")
gdf['demand_timeseries'] = gdf['demand_year0_kwh'].apply(
    lambda d: [d * (growth ** t) for t in range(T + 1)]
)
print(f"✓ demand_timeseries rebuilt for {gdf['demand_timeseries'].notna().sum():,} settlements")
print(f"  Sample: {gdf['demand_timeseries'].iloc[0][:3]}")

## 2. Cost Assumptions
Confirms all parameters for the SBEE expanded grid base scenario.

In [ ]:
print('=== BASE SCENARIO: SBEE NATIONAL EXPANSION PROGRAMME ===')
print()
print('GRID EXTENSION:')
print(f'  Connection cost          : $125/HH  (SBEE programme / ABERME 2022)')
print(f'  MV backbone              : public infrastructure (not in HH CAPEX)')
print(f'  Grid generation cost     : $0.08/kWh  (SBEE bulk tariff)')
print(f'  Max grid distance        : 0.45 degrees ≈ 50 km')
print(f'  Terrain penalty          : OnSSET 5-factor (slope/road/substation/landcover/elevation)')
print()
print('MINI-GRID:')
print(f'  Solar MG min demand      : 10,000 kWh/yr  (ESMAP lower bound)')
print(f'  Hybrid max road dist     : 100 km')
print(f'  Productive use uplift    : x1.30  (health/school/cropland settlements)')
print(f'  Solar CAPEX              : $2,500/kWp + $270/kWh LFP battery (6h)')
print(f'  Hydro CAPEX              : $4,500/kW')
print()
print('SHS:')
print(f'  Tier-2 kit               : $150/HH  (GOGLA West Africa 2023)')
print(f'  Tier-3 kit               : $350/HH  (mean_rwi >= 0.5)')
print()
print('DEMAND MODEL:')
print(f'  Source                   : VIDA satellite building footprints')
print(f'  Weights                  : small×0.217 + medium×0.650 + large×1.100 kWh/day')
print(f'  Intensity growth         : {DEMAND["demand_intensity_growth_rate"]*100:.1f}%/yr  (income-driven)')
print(f'  Population growth        : {DEMAND["population_growth_rate"]*100:.1f}%/yr x {DEMAND["rural_unelec_share"]*100:.0f}% rural')
print(f'  Combined growth          : ~4.1%/yr')
print(f'  Discount rate            : {DISCOUNT_RATE*100:.0f}%/yr')
print(f'  Planning horizon         : {HORIZON} years (2025-2040)')


## 3. Grid Routing — Option B: Nearest Electrified Settlement
Computes distance from each unelectrified settlement to the nearest currently-electrified settlement using cKDTree spatial index.

In [ ]:
from scipy.spatial import cKDTree

if 'lat' in gdf.columns and 'lon' in gdf.columns:
    elec_coords = gdf.loc[elec_mask, ['lat','lon']].values
    all_coords  = gdf[['lat','lon']].values
    tree        = cKDTree(elec_coords)
    dists, _    = tree.query(all_coords)
    gdf['dist_nearest_electrified_km'] = dists
    print('Option B routing complete ✓')
    print(f'  Electrified nodes used  : {elec_mask.sum():,}')
    print(f'  Unelectrified settlements: {unelec_mask.sum():,}')
    d = gdf.loc[unelec_mask, 'dist_nearest_electrified_km']
    print(f'  Distance (degrees):')
    print(f'    min={d.min():.4f}  median={d.median():.4f}  max={d.max():.4f}')
    print(f'    (1 degree ≈ 111 km — distances intentionally in degrees for SBEE model)')
else:
    gdf['dist_nearest_electrified_km'] = gdf.get('GridDistKm', 0.01)
    print('Warning: lat/lon not found — using GridDistKm fallback')


## 4. Grid LCOE — SBEE Expanded Programme
Uses the OnSSET `grid_price` methodology:
- **Infrastructure CAPEX** = LV connection only at $125/HH (MV backbone = public infra)
- **Grid generation cost** = $0.08/kWh (SBEE bulk tariff)
- **Terrain penalty** = OnSSET 5-factor multiplier
- **Max distance** = 0.45 degrees ≈ 50 km


In [ ]:
MAX_DIST_DEG          = 0.45   # degrees ≈ 50 km
#CONN_COST             = 125    # USD/HH — SBEE programme
CONN_COST             = 400    # USD/HH — High connection cost
GRID_GEN_COST_PER_KWH = 0.08   # USD/kWh — SBEE bulk tariff

def grid_lcoe_sbee(distance_km, num_households, demand_timeseries,
                   slope_deg=2.0, road_dist_km=10.0, substation_dist_km=5.0,
                   land_cover=14, elevation_m=300.0):
    """
    Grid LCOE for SBEE expanded programme.
    Infrastructure = LV connection only ($125/HH).
    MV backbone is public infrastructure — not charged to household LCOE.
    Includes grid generation cost ($0.08/kWh) alongside infrastructure amortisation.
    Source: ABERME Plan Directeur 2022; OnSSET grid_price methodology.
    """
    if distance_km < 0 or distance_km > MAX_DIST_DEG:
        return float('inf')
    if distance_km == 0:
        distance_km = 1e-6

    infra_capex = CONN_COST * num_households
    penalty     = compute_grid_penalty(slope_deg, road_dist_km,
                                       substation_dist_km, land_cover, elevation_m)
    infra_capex *= penalty
    annual_opex  = infra_capex * GRID['opex_rate']
    loss_factor  = 1 - GRID['loss_rate']

    ts = demand_timeseries if isinstance(demand_timeseries, list) else list(demand_timeseries)
    if not ts: return float('inf')

    energy_net = [e * loss_factor for e in ts]
    T = GRID['lifetime_years']
    if len(energy_net) < T + 1:
        energy_net = energy_net + [energy_net[-1]] * (T + 1 - len(energy_net))
    energy_net = [0.0] + energy_net[1:T+1]

    avg_energy = sum(energy_net[1:]) / T if T > 0 else 0
    annual_gen = avg_energy * GRID_GEN_COST_PER_KWH
    costs      = [infra_capex] + [annual_opex + annual_gen] * T
    nc         = npv_fn(costs,      DISCOUNT_RATE)
    ne         = npv_fn(energy_net, DISCOUNT_RATE)
    return nc / ne if ne > 0 else float('inf')

def _row_grid(r):
    ts = r.get('demand_timeseries', [])
    if isinstance(ts, str):
        try: ts = ast.literal_eval(ts)
        except: ts = []
    return grid_lcoe_sbee(
        distance_km        = float(r.get('dist_nearest_electrified_km') or 99),
        num_households     = float(r.get('num_households', 1) or 1),
        demand_timeseries  = ts,
        slope_deg          = float(r.get('Slope', 2.0) or 2.0),
        road_dist_km       = float(r.get('dist_road_km', 10.0) or 10.0),
        substation_dist_km = float(r.get('DistSubstation', 5.0) or 5.0),
        land_cover         = int(r.get('LandCover', 14) or 14),
        elevation_m        = float(r.get('Elevation', 300.0) or 300.0),
    )

print(f'Computing SBEE grid LCOE for {unelec_mask.sum():,} settlements...')
gdf.loc[unelec_mask, 'lcoe_grid']      = gdf[unelec_mask].apply(_row_grid, axis=1).values
gdf.loc[~unelec_mask, 'lcoe_grid']     = 0.0
gdf.loc[unelec_mask, 'grid_capex_usd'] = gdf.loc[unelec_mask, 'num_households'].fillna(1) * CONN_COST
gdf.loc[~unelec_mask, 'grid_capex_usd']= 0.0

feasible = (gdf.loc[unelec_mask, 'lcoe_grid'] < np.inf).sum()
fin = gdf.loc[unelec_mask, 'lcoe_grid'].replace(np.inf, np.nan).dropna()
print(f'  Feasible (within 50km): {feasible:,}')
print(f'  Median grid LCOE      : ${fin.median():.3f}/kWh')
print(f'  Min grid LCOE         : ${fin.min():.3f}/kWh')


## 5. Mini-Grid LCOE
Applies productive use uplift ×1.30 before feasibility check.
Solar MG threshold = 10,000 kWh/yr (ESMAP lower bound).


In [ ]:
PU_FACTOR = 1.30  # productive use uplift

gdf_mg_input = gdf[unelec_mask].copy()
gdf_mg_input['demand_year0_kwh'] = gdf_mg_input['demand_year0_kwh'] * PU_FACTOR
gdf_mg_input['demand_timeseries'] = gdf_mg_input['demand_timeseries'].apply(
    lambda ts: [x * PU_FACTOR for x in ts] if isinstance(ts, list) else ts
)

orig_threshold = MG_SUBTYPES['mg_solar']['min_demand_kwh']
MG_SUBTYPES['mg_solar']['min_demand_kwh'] = 10_000

print(f'Computing MG LCOE for {unelec_mask.sum():,} settlements...')
print(f'  Productive use uplift : x{PU_FACTOR}')
print(f'  Solar MG min demand   : {MG_SUBTYPES["mg_solar"]["min_demand_kwh"]:,} kWh/yr')
print(f'  Hydro MG max river    : {MG_SUBTYPES["mg_hydro"]["max_river_dist_km"]} km')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    gdf_mg = add_minigrid_lcoe(gdf_mg_input)

MG_SUBTYPES['mg_solar']['min_demand_kwh'] = orig_threshold

mg_cols = [c for c in gdf_mg.columns if 'lcoe_mg' in c or 'minigrid' in c.lower()]
for col in mg_cols:
    if col not in gdf.columns:
        gdf[col] = np.inf
    try:
        gdf.loc[unelec_mask, col] = gdf_mg[col].values
    except: pass

print(f'\n  Solar MG feasible : {(gdf.loc[unelec_mask,"lcoe_mg_solar"]<np.inf).sum():,}')
print(f'  Hydro MG feasible : {(gdf.loc[unelec_mask,"lcoe_mg_hydro"]<np.inf).sum():,}')
print(f'  Hybrid feasible   : {(gdf.loc[unelec_mask,"lcoe_mg_hybrid"]<np.inf).sum():,}')


## 6. SHS LCOE
Tier-2 ($150) for mean_rwi < 0.5, Tier-3 ($350) for mean_rwi ≥ 0.5.

In [ ]:
print(f'Computing SHS LCOE for {unelec_mask.sum():,} settlements...')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    gdf_shs = add_shs_lcoe(gdf[unelec_mask].copy())

for col in ['lcoe_shs', 'shs_capex_usd', 'shs_kit']:
    if col in gdf_shs.columns:
        if col not in gdf.columns:
            gdf[col] = np.inf
        try:
            gdf.loc[unelec_mask, col] = gdf_shs[col].values
        except: pass

fin_shs = gdf.loc[unelec_mask, 'lcoe_shs'].replace(np.inf, np.nan).dropna()
print(f'  Median SHS LCOE : ${fin_shs.median():.3f}/kWh')
print(f'  Tier-2 sites    : {(gdf.loc[unelec_mask,"shs_kit"]=="tier_2").sum():,}')
print(f'  Tier-3 sites    : {(gdf.loc[unelec_mask,"shs_kit"]=="tier_3").sum():,}')


In [ ]:
# Verify all LCOE columns present before selection
print("=== LCOE COLUMNS CHECK ===")
for col in ['lcoe_grid','lcoe_shs','lcoe_mg_solar','lcoe_mg_hydro','lcoe_mg_hybrid']:
    if col in gdf.columns:
        finite = (gdf.loc[unelec_mask, col] < np.inf).sum()
        med = gdf.loc[unelec_mask, col].replace(np.inf, np.nan).dropna()
        print(f"  {col:<20} {finite:>6,} finite  median=${med.median():.3f}/kWh" if len(med)>0 else f"  {col:<20} {finite:>6,} finite")
    else:
        print(f"  {col:<20} MISSING")

## 7. Least-Cost Technology Selection
Three-timestep electrification with grid creep (2030 / 2035 / 2040).
The SBEE grid LCOE is injected into the timestep loop to ensure consistency.


In [ ]:
import importlib
import src.least_cost.technology_selector as _ts_mod
importlib.reload(_ts_mod)
from src.least_cost.technology_selector import (
    select_least_cost, run_timestep_electrification,
    technology_summary, milestone_summary
)

# Force grid cap off
_ts_mod.GRID_CAP_SITES = 150

# Inject SBEE grid LCOE into timestep loop
def _sbee_add_grid_lcoe(df, dist_col='dist_nearest_electrified_km'):
    df = df.copy()
    def _row(r):
        dist = float(r.get(dist_col) or r.get('dist_nearest_electrified_km') or 99)
        if dist < 0 or dist > MAX_DIST_DEG: return float('inf')
        if dist == 0: dist = 1e-6
        ts = r.get('demand_timeseries', [])
        if isinstance(ts, str):
            try: ts = ast.literal_eval(ts)
            except: ts = []
        if not ts: return float('inf')
        infra = CONN_COST * float(r.get('num_households', 1) or 1)
        pen   = compute_grid_penalty(
            float(r.get('Slope', 2.0) or 2.0),
            float(r.get('dist_road_km', 10.0) or 10.0),
            float(r.get('DistSubstation', 5.0) or 5.0),
            int(r.get('LandCover', 14) or 14),
            float(r.get('Elevation', 300.0) or 300.0))
        infra *= pen
        loss  = 1 - GRID['loss_rate']
        T     = GRID['lifetime_years']
        en    = [e * loss for e in ts]
        if len(en) < T + 1: en = en + [en[-1]] * (T + 1 - len(en))
        en    = [0.0] + en[1:T+1]
        avg_e = sum(en[1:]) / T if T > 0 else 0
        costs = [infra] + [infra * GRID['opex_rate'] + avg_e * GRID_GEN_COST_PER_KWH] * T
        nc    = npv_fn(costs, DISCOUNT_RATE)
        ne    = npv_fn(en,    DISCOUNT_RATE)
        return nc / ne if ne > 0 else float('inf')
    df['lcoe_grid']      = df.apply(_row, axis=1)
    df['grid_capex_usd'] = df['num_households'].fillna(1) * CONN_COST
    return df

_ts_mod.add_grid_lcoe = _sbee_add_grid_lcoe
print('✓ SBEE grid LCOE injected into timestep loop')
print(f'✓ GRID_CAP_SITES = {_ts_mod.GRID_CAP_SITES}')

# Step 1: Initial selection
gdf = select_least_cost(gdf)

# Step 2: Three-timestep electrification
gdf = run_timestep_electrification(gdf, verbose=False)

# Step 3: Reassign low-demand grid sites to SHS
# Settlements below 1,350 kWh/yr are too small for grid even at $125/HH
DEMAND_THRESHOLD = 1350  # kWh/yr
low_demand_grid  = (
    (gdf['least_cost_tech'] == 'Grid Extension') &
    (gdf['demand_year0_kwh'].fillna(0) < DEMAND_THRESHOLD)
)
print(f'\nReassigning {low_demand_grid.sum():,} low-demand grid sites to SHS')
gdf.loc[low_demand_grid, 'least_cost_tech']  = 'SHS'
gdf.loc[low_demand_grid, 'least_cost_capex'] = (
    gdf.loc[low_demand_grid, 'num_households'].fillna(1) * 150
)

# Step 4: Reassign excess hydro → solar MG (keep best 28 hydro sites)
hydro_idx  = gdf[gdf['least_cost_tech'] == 'Mini-Grid: Mini-Hydro'].index
hydro_keep = gdf.loc[hydro_idx, 'lcoe_mg_hydro'].nsmallest(28).index
hydro_move = hydro_idx.difference(hydro_keep)
if len(hydro_move) > 0:
    print(f'Reassigning {len(hydro_move):,} excess hydro → Solar MG')
    gdf.loc[hydro_move, 'least_cost_tech']  = 'Mini-Grid: Solar PV Only'
    gdf.loc[hydro_move, 'least_cost_capex'] = (
        gdf.loc[hydro_move, 'minigrid_capex_usd'].fillna(
            gdf.loc[hydro_move, 'num_households'].fillna(1) * 280
        )
    )

print('\n=== TECHNOLOGY DISTRIBUTION ===')
print(technology_summary(gdf).to_string(index=False))
print(f"\nTotal CAPEX: ${gdf['least_cost_capex'].sum()/1e6:.1f}M")


## 8. CAPEX Reconciliation
Fixes CAPEX columns to reflect correct unit costs per technology.

In [ ]:
# Grid CAPEX = $125/HH connection cost
grid_mask = gdf['least_cost_tech'] == 'Grid Extension'
gdf.loc[grid_mask, 'least_cost_capex'] = (
    gdf.loc[grid_mask, 'num_households'].fillna(1) * 125
)

# SHS CAPEX = $150/HH Tier-2 (default)
shs_mask = gdf['least_cost_tech'] == 'SHS'
gdf.loc[shs_mask, 'least_cost_capex'] = (
    gdf.loc[shs_mask, 'num_households'].fillna(1) * 150
)

# MG CAPEX from minigrid_capex_usd (computed by add_minigrid_lcoe)
mg_mask = gdf['least_cost_tech'].isin([
    'Mini-Grid: Solar PV Only',
    'Mini-Grid: Mini-Hydro',
    'Mini-Grid: Solar-Diesel Hybrid'
])
if 'minigrid_capex_usd' in gdf.columns:
    gdf.loc[mg_mask, 'least_cost_capex'] = (
        gdf.loc[mg_mask, 'minigrid_capex_usd'].fillna(
            gdf.loc[mg_mask, 'num_households'].fillna(1) * 400
        )
    )

print('=== FINAL CAPEX BREAKDOWN ===')
total = 0
for tech in ['Grid Extension','SHS','Mini-Grid: Solar PV Only',
             'Mini-Grid: Mini-Hydro','Already Electrified']:
    mask  = gdf['least_cost_tech'] == tech
    capex = gdf.loc[mask, 'least_cost_capex'].sum()
    sites = mask.sum()
    hh    = gdf.loc[mask, 'num_households'].sum()
    total += capex
    print(f'  {tech:<35} {sites:>6,} sites  {hh:>9,.0f} HH  ${capex/1e6:.1f}M')
print(f'  {"─"*70}')
print(f'  {"TOTAL":<35} {len(gdf):>6,} sites  {gdf["num_households"].sum():>9,.0f} HH  ${total/1e6:.1f}M')


## 9. Technology Map

In [ ]:
import matplotlib.patches as mpatches

TECH_COLORS = {
    'Already Electrified'            : '#BDBDBD',
    'Grid Extension'                 : '#1565C0',
    'Mini-Grid: Solar PV Only'       : '#F9A825',
    'Mini-Grid: Solar-Diesel Hybrid' : '#E53935',
    'Mini-Grid: Mini-Hydro'          : '#00897B',
    'SHS'                            : '#43A047',
}
TECH_SIZES = {
    'Already Electrified': 3, 'Grid Extension': 8,
    'Mini-Grid: Solar PV Only': 5, 'Mini-Grid: Mini-Hydro': 5,
    'Mini-Grid: Solar-Diesel Hybrid': 5, 'SHS': 2,
}

fig, ax = plt.subplots(figsize=(9, 11))
ax.set_facecolor('#F5F5F5')

for bp in [Path('..')/'data'/'raw'/'Benin_boundary.gpkg',
           Path('..')/'data'/'raw'/'benin_boundary.gpkg']:
    if bp.exists():
        gpd.read_file(bp).plot(ax=ax, color='white', edgecolor='#333',
                               linewidth=1.2, zorder=1)
        break

tp = Path('..')/'data'/'raw'/'Benin_existing_transmission_lines_2017.geojson'
if tp.exists():
    gpd.read_file(tp).plot(ax=ax, color='#212121', linewidth=0.9, alpha=0.7, zorder=2)

plot_order = ['SHS','Mini-Grid: Solar-Diesel Hybrid','Mini-Grid: Solar PV Only',
              'Mini-Grid: Mini-Hydro','Grid Extension','Already Electrified']
for tech in plot_order:
    sub = gdf[gdf['least_cost_tech'] == tech]
    if len(sub) == 0: continue
    sub.plot(ax=ax, color=TECH_COLORS.get(tech,'#999'), markersize=TECH_SIZES.get(tech,3),
             alpha=0.75, marker='o', linewidth=0, zorder=3)

ax.set_xlim(0.8, 3.9); ax.set_ylim(6.1, 12.5)
counts = gdf['least_cost_tech'].value_counts()
leg = []
for t in plot_order:
    n = counts.get(t, 0)
    if n > 0:
        leg.append(mpatches.Patch(color=TECH_COLORS.get(t,'#999'),
                   label=f'{t} ({n:,} — {n/len(gdf)*100:.1f}%)'))
leg.append(mpatches.Patch(color='#212121', label='Transmission lines'))
ax.legend(handles=leg, loc='lower left', fontsize=7.5, framealpha=0.95,
          title='Technology', title_fontsize=8)
ax.set_title(f'Benin — Least-Cost Electrification (SBEE Base Scenario)\n'
             f'{len(gdf):,} settlements | ${{gdf["least_cost_capex"].sum()/1e6:.1f}}M total CAPEX',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()

maps_dir = Path('..') / 'data' / 'outputs' / 'maps'
maps_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(maps_dir / 'technology_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Map saved ✓')


## 10. Save Results
Saves all outputs: GeoJSON, summary CSVs, and Streamlit CSV.

In [ ]:
from datetime import datetime

ts        = datetime.now().strftime('%Y%m%d_%H%M')
TABLE_DIR = Path('..') / 'data' / 'outputs' / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Fix CAPEX
grid_mask = gdf['least_cost_tech'] == 'Grid Extension'
shs_mask  = gdf['least_cost_tech'] == 'SHS'
mg_mask   = gdf['least_cost_tech'].isin(['Mini-Grid: Solar PV Only','Mini-Grid: Mini-Hydro'])

gdf.loc[grid_mask, 'least_cost_capex'] = gdf.loc[grid_mask, 'num_households'].fillna(1) * 125
gdf.loc[shs_mask,  'least_cost_capex'] = gdf.loc[shs_mask,  'num_households'].fillna(1) * 150
if 'minigrid_capex_usd' in gdf.columns:
    gdf.loc[mg_mask, 'least_cost_capex'] = gdf.loc[mg_mask, 'minigrid_capex_usd'].fillna(
        gdf.loc[mg_mask, 'num_households'].fillna(1) * 400)

# Save Streamlit CSV
COLS = {
    'elec_status':'ElecStatus','least_cost_tech':'MinimumOverall',
    'least_cost_capex':'InvestmentCost','least_cost_lcoe':'MinimumOverallLCOE',
    'num_households':'NumConnections','demand_year0_kwh':'DemandKWh_Y0',
    'dist_nearest_electrified_km':'DistNearElecKm',
    'ElecTargetPhase':'ElecTargetPhase','ElecTargetYear':'ElecTargetYear',
    'GridRolloutPhase':'GridRolloutPhase',
    'lcoe_grid':'LCOE_Grid','lcoe_shs':'LCOE_SHS',
    'lcoe_mg_solar':'LCOE_MG_Solar','lcoe_mg_hydro':'LCOE_MG_Hydro',
    'lat':'Lat','lon':'Lon',
}
df_st   = gdf.rename(columns={k:v for k,v in COLS.items() if k in gdf.columns})
st_cols = [v for v in COLS.values() if v in df_st.columns]
csv_path = TABLE_DIR / f'benin_electrification_streamlit_{ts}.csv'
df_st[st_cols].to_csv(csv_path, index=False)
print(f'✓ Saved: {csv_path.name}')

print(f'\n=== FINAL RESULTS ===')
for tech in ['Grid Extension','SHS','Mini-Grid: Solar PV Only',
             'Mini-Grid: Mini-Hydro','Already Electrified']:
    mask  = gdf['least_cost_tech'] == tech
    capex = gdf.loc[mask,'least_cost_capex'].sum()
    sites = mask.sum()
    hh    = gdf.loc[mask,'num_households'].sum()
    print(f'  {tech:<35} {sites:>6,} sites  {hh:>9,.0f} HH  ${capex/1e6:.1f}M')


from pathlib import Path
import geopandas as gpd
from datetime import datetime

ts      = datetime.now().strftime('%Y%m%d_%H%M')
OUT_DIR = Path('..') / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

save_geo = [c for c in gdf.columns if c != 'demand_timeseries']
gdf[save_geo].to_file(OUT_DIR / f'settlements_lcoe_{ts}.geojson', driver='GeoJSON')
print(f'✓ GeoJSON saved: settlements_lcoe_{ts}.geojson')
print(f'  Size: {(OUT_DIR / f"settlements_lcoe_{ts}.geojson").stat().st_size/1e6:.1f} MB')

In [ ]:
# ── Add coordinate columns for Streamlit map ─────────────────────────────
gdf['X_deg'] = gdf['lon'] if 'lon' in gdf.columns else gdf.geometry.centroid.x
gdf['Y_deg'] = gdf['lat'] if 'lat' in gdf.columns else gdf.geometry.centroid.y

# ── Save Streamlit CSV ────────────────────────────────────────────────────
from datetime import datetime
from pathlib import Path

ts        = datetime.now().strftime('%Y%m%d_%H%M')
TABLE_DIR = Path('..') / 'data' / 'outputs' / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)

save_cols = [c for c in [
    'elec_status','least_cost_tech','least_cost_capex','least_cost_lcoe',
    'num_households','demand_year0_kwh','dist_nearest_electrified_km',
    'ElecTargetPhase','ElecTargetYear','GridRolloutPhase',
    'lcoe_grid','lcoe_shs','lcoe_mg_solar','lcoe_mg_hydro',
    'X_deg','Y_deg','dist_road_km'
] if c in gdf.columns]

df_st = gdf[save_cols].copy().rename(columns={
    'elec_status'                 : 'ElecStatus',
    'least_cost_tech'             : 'MinimumOverall',
    'least_cost_capex'            : 'InvestmentCost',
    'least_cost_lcoe'             : 'MinimumOverallLCOE',
    'num_households'              : 'NumConnections',
    'demand_year0_kwh'            : 'DemandKWh_Y0',
    'dist_nearest_electrified_km' : 'DistNearElecKm',
    'lcoe_grid'                   : 'LCOE_Grid',
    'lcoe_shs'                    : 'LCOE_SHS',
    'lcoe_mg_solar'               : 'LCOE_MG_Solar',
    'lcoe_mg_hydro'               : 'LCOE_MG_Hydro',
    'dist_road_km'                : 'DistRoadKm',
})

csv_path = TABLE_DIR / f'benin_electrification_streamlit_{ts}.csv'
df_st.to_csv(csv_path, index=False)

print(f'✓ Saved: {csv_path.name}')
print(f'  Rows: {len(df_st):,}  |  Columns: {len(df_st.columns)}')
print(f'\n=== FINAL RESULTS ===')
for tech in ['Grid Extension','SHS','Mini-Grid: Solar PV Only',
             'Mini-Grid: Mini-Hydro','Already Electrified']:
    mask  = gdf['least_cost_tech'] == tech
    capex = gdf.loc[mask,'least_cost_capex'].sum()
    sites = mask.sum()
    print(f'  {tech:<35} {sites:>6,} sites  ${capex/1e6:.1f}M')